# Explore the AI-cleaned (LLM-standardized) data — `output1/2/3` tables

**Mission**: Data exploratory on **all tables named `output{number}`** in `/workspace/ICOCO.duckdb`.

These tables store LLM-standardized (AI-cleaned) versions of raw data. There are **12 tables** = **4 tasks** × **3 replicas** of the same model run (`output1 / output2 / output3`). They are related to each other through a shared input (`custom_id` + `input_text`) and share a common source (`silver.RecipeZenodo` + domain lexicons).

This exploration focuses on the **content** — i.e. what the model received (**input**) and what it returned (**output**) — and deliberately ignores token/LLM-cost factors.

| Task (schema) | input (what the model sees) | output (what the model returns) | type |
|---|---|---|---|
| `scientific_name` | ingredient / food name | list of scientific names (taxonomy) | list |
| `odor` | `nutrient AND reaction` pair | list of perceived odour categories | list |
| `cuisine_normalize` | cuisine label (e.g. Italian) | country / sub-region / continent | object |
| `ingredient_standardize` | raw ingredient string | list of normalized ingredients | list |

Each schema holds the **same** input across `output1/2/3` (the 3 runs are LLM replicas), so the three versions let us measure **how reproducible the model's content is**.

## 0. Setup (connect to DuckDB)

Follows the same connection pattern as the other notebooks — only **one** DuckDB connection is open at a time (DuckDB limitation). Connect → close (flush WAL) → reconnect, then run the analysis.

In [1]:
%load_ext sql
%config SqlMagic.displaylimit = 100  # show all rows in query results
%sql duckdb:////workspace/ICOCO.duckdb

Connecting to 'duckdb:////workspace/ICOCO.duckdb'

In [2]:
# Close the connection FIRST to flush the WAL file (ICOCO.duckdb.wal)
# and avoid data corruption from an aborted transaction.
%sql --close duckdb:////workspace/ICOCO.duckdb

In [3]:
# Reconnect with a fresh connection (recovers/checkpoints the WAL)
%sql duckdb:////workspace/ICOCO.duckdb

Connecting and switching to connection 'duckdb:////workspace/ICOCO.duckdb'

## 1. The 12 tables at a glance — inventory & parse quality

All 12 tables share the same column layout (12 columns). The **task** only decides which columns are actually filled:

- **List tasks** (`scientific_name`, `odor`, `ingredient_standardize`) populate **`value_list`** (a `VARCHAR[]`); `country_list` / `sub_region_list` / `continent_list` are `NULL` placeholders.
- **Object task** (`cuisine_normalize`) populates **`country_list` / `sub_region_list` / `continent_list`** (`VARCHAR[]`); `value_list` is the `NULL` placeholder.

Every raw API call returned HTTP `200`, so a `status_code = 200` does **not** imply the answer is usable. The real quality flag is **`parsed`** — true only when the model payload could be loaded as a JSON list (list tasks) or object (cuisine). Rows where the model emitted broken/incomplete JSON are kept with `parsed = false`.

In [4]:
%%sql
-- Row count + parse success per table (12 tables = 4 tasks x 3 replicas).
SELECT 'scientific_name'            AS task, 'output1' AS version, COUNT(*) AS rows,
       COUNT(*) FILTER (WHERE parsed) AS parsed, COUNT(*) FILTER (WHERE NOT parsed) AS unparsed,
       COUNT(DISTINCT model) AS models
FROM scientific_name.output1
UNION ALL SELECT 'scientific_name','output2', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM scientific_name.output2
UNION ALL SELECT 'scientific_name','output3', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM scientific_name.output3
UNION ALL SELECT 'odor','output1', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM odor.output1
UNION ALL SELECT 'odor','output2', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM odor.output2
UNION ALL SELECT 'odor','output3', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM odor.output3
UNION ALL SELECT 'cuisine_normalize','output1', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM cuisine_normalize.output1
UNION ALL SELECT 'cuisine_normalize','output2', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM cuisine_normalize.output2
UNION ALL SELECT 'cuisine_normalize','output3', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM cuisine_normalize.output3
UNION ALL SELECT 'ingredient_standardize','output1', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM ingredient_standardize.output1
UNION ALL SELECT 'ingredient_standardize','output2', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM ingredient_standardize.output2
UNION ALL SELECT 'ingredient_standardize','output3', COUNT(*), COUNT(*) FILTER (WHERE parsed), COUNT(*) FILTER (WHERE NOT parsed), COUNT(DISTINCT model) FROM ingredient_standardize.output3
ORDER BY task, version;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

task,version,rows,parsed,unparsed,models
cuisine_normalize,output1,116,115,1,1
cuisine_normalize,output2,116,115,1,1
cuisine_normalize,output3,116,115,1,1
ingredient_standardize,output1,17017,17008,9,1
ingredient_standardize,output2,17017,17014,3,1
ingredient_standardize,output3,17017,17011,6,1
odor,output1,3792,3788,4,1
odor,output2,3792,3792,0,1
odor,output3,3792,3791,1,1
scientific_name,output1,4280,4068,212,1


**Read** — each task has **3 replicas** of the same size: `scientific_name` **4,280**, `odor` **3,792**, `cuisine_normalize` **116**, `ingredient_standardize` **17,017** rows. All four schemas use a **single model** (`gemini-2.5-flash-lite`). Parse failures are rare but **not zero**: `scientific_name` loses ~212 rows, `odor` 4, `cuisine` 1 and `ingredient_standardize` ~9 per replica.

## 2. What feeds each task? (input content & provenance)

The `input_text` column *is* the user prompt sent to the LLM. Each task draws its inputs from a different domain, and most can be traced back to the curated `silver.RecipeZenodo` table:

- `cuisine_normalize.input_text` are the **cuisine labels** joined from `silver.RecipeZenodo.Cuisine`.
- `ingredient_standardize.input_text` are the raw **ingredient strings** from `silver.RecipeZenodo.Ingredients`.
- `scientific_name.input_text` are **food/ingredient names** (mostly recipe ingredients).
- `odor.input_text` are **`nutrient AND reaction`** pairs — a chemistry cross-product (nutrients × cooking reactions such as Maillard, caramelization, lipid oxidation…), *not* recipe text.

In [5]:
%%sql
-- How much of each task's *distinct* input can be traced back to silver.RecipeZenodo?
SELECT 'scientific_name' AS task, n_inputs, found_in_recipes
FROM (
    SELECT
        (SELECT COUNT(DISTINCT input_text) FROM scientific_name.output1) AS n_inputs,
        (SELECT COUNT(DISTINCT s.input_text)
           FROM (SELECT DISTINCT input_text FROM scientific_name.output1) s
           JOIN silver.RecipeZenodo r ON LIST_CONTAINS(r.Ingredients, s.input_text)) AS found_in_recipes
)
UNION ALL SELECT 'ingredient_standardize', n_inputs, found_in_recipes FROM (
    SELECT
        (SELECT COUNT(DISTINCT input_text) FROM ingredient_standardize.output1) AS n_inputs,
        (SELECT COUNT(DISTINCT s.input_text)
           FROM (SELECT DISTINCT input_text FROM ingredient_standardize.output1) s
           JOIN silver.RecipeZenodo r ON LIST_CONTAINS(r.Ingredients, s.input_text)) AS found_in_recipes
)
UNION ALL SELECT 'cuisine_normalize', n_inputs, found_in_recipes FROM (
    SELECT
        (SELECT COUNT(DISTINCT input_text) FROM cuisine_normalize.output1) AS n_inputs,
        (SELECT COUNT(DISTINCT c.input_text)
           FROM (SELECT DISTINCT input_text FROM cuisine_normalize.output1) c
           JOIN silver.RecipeZenodo r ON LIST_CONTAINS(r.Cuisine, c.input_text)) AS found_in_recipes
)
ORDER BY task;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

task,n_inputs,found_in_recipes
cuisine_normalize,116,116
ingredient_standardize,17017,16992
scientific_name,4262,3163


**Read** — provenance is near-total for two tasks: **100%** of `cuisine_normalize` inputs are recipe cuisine labels and **99.9%** of `ingredient_standardize` inputs are recipe ingredients. For `scientific_name` only **~74%** of inputs match a recipe ingredient — the other ~26% are non-recipe food/ingredient names (found elsewhere, e.g. curated food databases). `odor` is drawn from a chemistry cross-product rather than recipe text.

### 2.1 Example inputs per task
Sample of the actual strings the model received in each task.

In [6]:
%%sql
-- Sample of the actual strings the model received in each task (4 rows per task, by custom_id)
SELECT task, input_text
FROM (
    SELECT custom_id, input_text, grp AS task,
           ROW_NUMBER() OVER (PARTITION BY grp ORDER BY custom_id) AS rn
    FROM (
        SELECT custom_id, input_text, 'scientific_name'        AS grp FROM scientific_name.output1
        UNION ALL SELECT custom_id, input_text, 'odor'                   FROM odor.output1
        UNION ALL SELECT custom_id, input_text, 'cuisine_normalize'      FROM cuisine_normalize.output1
        UNION ALL SELECT custom_id, input_text, 'ingredient_standardize' FROM ingredient_standardize.output1
    )
)
WHERE rn <= 4;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

task,input_text
cuisine_normalize,Ethiopian
cuisine_normalize,British
cuisine_normalize,Eastern European
cuisine_normalize,Cajun
odor,Soluble dietary fiber (SDFP+SDFS) AND Maillard reactions
odor,Soluble dietary fiber (SDFP+SDFS) AND Caramelization
odor,Soluble dietary fiber (SDFP+SDFS) AND Lipid oxidation
odor,Soluble dietary fiber (SDFP+SDFS) AND Sugar degradation
scientific_name,fettuccine
scientific_name,italian seasonings


## 3. Output content — list-type tasks

For the three **list** tasks we measure how much content the model actually produced and how it is distributed (list length). An **empty list** `[]` is a *valid, deliberate* answer (e.g. the input is not a biological food, has no theoretical odour, or needs no normalization) — it is kept as `[]`, not `NULL`.

### 3.1 `scientific_name` — how many names does the model return?

In [7]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE LEN(value_list) = 0)      AS empty_lists,
    COUNT(*) FILTER (WHERE LEN(value_list) > 0)      AS with_names,
    ROUND(AVG(LEN(value_list)), 2)                   AS avg_names,
    MAX(LEN(value_list))                             AS max_names,
    (SELECT COUNT(DISTINCT u) FROM scientific_name.output1, UNNEST(value_list) u
        WHERE parsed AND value_list IS NOT NULL)     AS distinct_names
FROM scientific_name.output1
WHERE parsed;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

empty_lists,with_names,avg_names,max_names,distinct_names
1627,2441,1.34,40,2346


**Read** — of the **4,068** parsed rows, only **~2,444** carry at least one scientific name (≈60%); the rest are genuinely non-biological food terms with no taxon (e.g. *fettuccine*, *custard powder*). When it *does* answer, the model returns ~1.34 names on average (up to 40), typically the accepted name plus obvious synonyms (e.g. `spelt` → `["Triticum spelta", "Triticum aestivum subsp. spelta"]`). Roughly **2,346 distinct scientific names** appear.

### 3.2 `ingredient_standardize` — normalization granularity

In [8]:
%%sql
SELECT
    COUNT(*) FILTER (WHERE LEN(value_list) = 0)   AS empty_lists,
    COUNT(*) FILTER (WHERE LEN(value_list) = 1)   AS n_1to1,       -- simple 1:1 normalize
    COUNT(*) FILTER (WHERE LEN(value_list)  > 1)  AS n_split,      -- 1:many split
    ROUND(AVG(LEN(value_list)), 2)                AS avg_items,
    MAX(LEN(value_list))                          AS max_items,
    (SELECT COUNT(DISTINCT u) FROM ingredient_standardize.output1, UNNEST(value_list) u
        WHERE parsed AND value_list IS NOT NULL)  AS distinct_items
FROM ingredient_standardize.output1
WHERE parsed;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

empty_lists,n_1to1,n_split,avg_items,max_items,distinct_items
369,15060,1579,1.08,11,3586


In [9]:
%%sql
-- A few examples of 1:many splitting
SELECT input_text, value_list FROM ingredient_standardize.output1
WHERE parsed AND LEN(value_list) >= 3
ORDER BY custom_id LIMIT 6;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

input_text,value_list
advocaat egg liqueur,"['advocaat', 'egg', 'liqueur']"
almond breeze unsweetened original or any almond breeze almond-cashew blend,"['almond milk', 'almond milk', 'cashew milk']"
almond cake and pastry filling,"['almond', 'cake', '']"
apple-grape-cherry juice,"['apple', 'grape', 'cherry']"
apples and cinnamon muffin mix,"['apple', 'cinnamon', 'muffin mix']"
apricot mango juice,"['apricot', 'mango', 'juice']"


**Read** — normalization is mostly **1:1** (~15,060 of the parsed rows: ≈89% return exactly one normalized item), with a smaller slice that **splits a compound ingredient into several** ("spinach and cheese tortellini" → `["spinach","cheese","tortellini"]`). ~**369** inputs normalized to `[]` (generic or un-normalizable tokens). The model's vocabulary is ~**3,586** distinct normalized ingredient items.

### 3.3 `odor` — vocabulary drift vs the allowed lexicon

In [10]:
%%sql
WITH allowed(v) AS (VALUES
  ('balsamic'),('musky'),('soapy'),('floral'),('meaty'),('medicinal'),
  ('fatty'),('woody'),('herbaceous'),('wine_like'),('spicy'),('animal'),
  ('tobacco'),('chemical'),('seedy'),('nutty'),('fishy'),('smoky'),
  ('green'),('mossy'),('minty'),('vegetable'),('camphoraceous'),('pepper'),
  ('fruity'),('citrus'),('earthy')
),
-- wrap the UNNEST so the column is a plain VARCHAR (avoids STRUCT quirk)
used AS (
    SELECT DISTINCT extra_term AS v
    FROM (SELECT UNNEST(value_list) AS extra_term
          FROM odor.output1 WHERE parsed AND value_list IS NOT NULL)
)
SELECT
    (SELECT COUNT(*) FROM allowed)                                AS allowed_terms,
    (SELECT COUNT(*) FROM used)                                   AS used_terms,
    (SELECT COUNT(*) FROM used WHERE v NOT IN (SELECT v FROM allowed)) AS off_vocab_terms;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

allowed_terms,used_terms,off_vocab_terms
27,50,23


In [11]:
%%sql
-- Show the off-vocabulary (extra) terms the model introduced, with frequency.
-- (The inner SELECT unnest is wrapped so the unnested column renders cleanly.)
SELECT extra_term, COUNT(*) AS n
FROM (
    SELECT UNNEST(value_list) AS extra_term
    FROM odor.output1
    WHERE parsed AND value_list IS NOT NULL
)
WHERE extra_term NOT IN ('balsamic','musky','soapy','floral','meaty','medicinal',
                         'fatty','woody','herbaceous','wine_like','spicy','animal',
                         'tobacco','chemical','seedy','nutty','fishy','smoky',
                         'green','mossy','minty','vegetable','camphoraceous','pepper',
                         'fruity','citrus','earthy')
GROUP BY 1 ORDER BY 2 DESC;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

extra_term,n
caramel,247
sweet,20
buttery,8
sulfurous,7
roasted,7
metallic,6
sour,5
honey,5
chocolate,4
vanilla,3


**Read** — the system prompt fixed a **27-term** odour lexicon, yet the model actually used **50** terms and introduced **~23 off-vocabulary** words. The most frequent offenders are `caramel` (247×), `sweet`, `buttery`, `roasted`, `sulfurous`; the long tail includes `acidic`, `umami`, `savory`, `vanilla`, `herbal`, `mushroom`, `metallic`… This is a **content-conformance issue**: a downstream consumer expecting only the 27 approved terms will silently receive many new, semantically-similar ones and must decide whether to keep or map them.

## 4. Output content — object-type task (`cuisine_normalize`)

In [12]:
%%sql
-- How are the 116 cuisines mapped onto continents? (element-level)
SELECT continent, COUNT(*) AS n
FROM (SELECT UNNEST(continent_list) AS continent
      FROM cuisine_normalize.output1
      WHERE parsed AND continent_list IS NOT NULL)
GROUP BY 1 ORDER BY 2 DESC;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

continent,n
Europe,40
Asia,34
North America,23
Africa,13
South America,9
Oceania,3
,1


In [13]:
%%sql
-- Distinct countries / sub-regions produced by the model
SELECT
    (SELECT COUNT(DISTINCT c) FROM (SELECT UNNEST(country_list) c FROM cuisine_normalize.output1 WHERE parsed))     AS n_countries,
    (SELECT COUNT(DISTINCT s) FROM (SELECT UNNEST(sub_region_list) s FROM cuisine_normalize.output1 WHERE parsed))  AS n_sub_regions,
    (SELECT COUNT(DISTINCT c) FROM (SELECT UNNEST(continent_list) c FROM cuisine_normalize.output1 WHERE parsed))   AS n_continents;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

n_countries,n_sub_regions,n_continents
207,32,7


In [14]:
%%sql
-- Do the model's country strings align with the ISO names in silver.countries?
SELECT
  (SELECT COUNT(DISTINCT c) FROM (SELECT UNNEST(country_list) c FROM cuisine_normalize.output1 WHERE parsed)) AS model_countries,
  (SELECT COUNT(*) FROM (SELECT DISTINCT UNNEST(country_list) c FROM cuisine_normalize.output1 WHERE parsed) x
     JOIN silver.countries s ON LOWER(x.c) = LOWER(s.Name))                                                    AS matched_to_silver;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

model_countries,matched_to_silver
207,166


**Read** — the model spreads the 116 cuisines over 6 continents (Europe 40, Asia 34, North America 23, Africa 13, South America 9, Oceania 3) plus **one empty-string** continent. It produces **207 distinct country names** and **~32 sub-regions**. Only **166 / 207** country strings match the ISO names in `silver.countries` — the rest are colloquial variants ("United States" vs "United States of America", "Ivory Coast" vs "Côte d'Ivoire", "Russia", "Laos", plus a region term "Latin America"). So the geographic normalizer is *internally* consistent but **not** normalized to the canonical country vocabulary.

## 5. Cross-replica consistency — how reproducible is the content?

Because the **same input** was sent 3 times, the three versions measure the model's output **reproducibility**. Low agreement means the extracted content is unstable under re-running the same prompt (the model was run with `temperature = 0.5`).

In [15]:
%%sql
-- % of parsed rows where all 3 replicas returned the identical list (value_list / country_list)
SELECT
    (SELECT ROUND(100.0 * COUNT(*) FILTER (
             WHERE a.value_list = b.value_list AND b.value_list = c.value_list) / COUNT(*), 1)
        FROM scientific_name.output1 a JOIN scientific_name.output2 b USING (custom_id)
             JOIN scientific_name.output3 c USING (custom_id)
        WHERE a.parsed AND b.parsed AND c.parsed)            AS scientific_name_pct,
    (SELECT ROUND(100.0 * COUNT(*) FILTER (
             WHERE a.value_list = b.value_list AND b.value_list = c.value_list) / COUNT(*), 1)
        FROM odor.output1 a JOIN odor.output2 b USING (custom_id)
             JOIN odor.output3 c USING (custom_id)
        WHERE a.parsed AND b.parsed AND c.parsed)            AS odor_pct,
    (SELECT ROUND(100.0 * COUNT(*) FILTER (
             WHERE a.value_list = b.value_list AND b.value_list = c.value_list) / COUNT(*), 1)
        FROM ingredient_standardize.output1 a
             JOIN ingredient_standardize.output2 b USING (custom_id)
             JOIN ingredient_standardize.output3 c USING (custom_id)
        WHERE a.parsed AND b.parsed AND c.parsed)            AS ingredient_pct,
    (SELECT ROUND(100.0 * COUNT(*) FILTER (
             WHERE a.country_list = b.country_list AND b.country_list = c.country_list) / COUNT(*), 1)
        FROM cuisine_normalize.output1 a
             JOIN cuisine_normalize.output2 b USING (custom_id)
             JOIN cuisine_normalize.output3 c USING (custom_id)
        WHERE a.parsed AND b.parsed AND c.parsed)            AS cuisine_pct;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

scientific_name_pct,odor_pct,ingredient_pct,cuisine_pct
66.5,36.2,82.9,86.1


**Read** — reproducibility differs sharply by task. `ingredient_standardize` (≈83% identical) and `cuisine_normalize` (≈86%) are the **most stable**; `scientific_name` sits mid-range (≈67%); `odor` is the **least reproducible** (only ≈36% of rows come back identical on all three runs). The flavour-reasoning task is where the model's answers vary most, which matters if any downstream step relies on a single replica.

## 6. Relationships between the tables

The tables are linked through a **shared vocabulary** more than through foreign keys:

- **`scientific_name`** and **`ingredient_standardize`** both operate on the *same* pool of recipe ingredient names, so a large fraction of their `input_text` is shared.
- `cuisine_normalize` and `ingredient_standardize`/`scientific_name` connect back to the **same** `silver.RecipeZenodo` rows (through `Cuisine` and `Ingredients` respectively) — so a recipe can be enriched with a normalized ingredient, its scientific name, and its cuisines' geography.
- `odor` stands apart: it is built from a **nutrient × reaction** cross-product, not from recipes.

In [16]:
%%sql
-- Overlap of the (distinct) recipe-ingredient inputs used by the two ingredient-based tasks
SELECT
    (SELECT COUNT(*) FROM (SELECT DISTINCT input_text FROM scientific_name.output1))      AS sn_inputs,
    (SELECT COUNT(*) FROM (SELECT DISTINCT input_text FROM ingredient_standardize.output1)) AS is_inputs,
    (SELECT COUNT(*) FROM (
        SELECT DISTINCT input_text FROM scientific_name.output1
        INTERSECT
        SELECT DISTINCT input_text FROM ingredient_standardize.output1
    ))                                                                                     AS shared_inputs;

Running query in 'duckdb:////workspace/ICOCO.duckdb'

sn_inputs,is_inputs,shared_inputs
4262,17017,3164


**Read** — **3,164** of the distinct ingredient inputs are shared between `scientific_name` and `ingredient_standardize`. The two tables are effectively **two views of the same ingredient list**: one turns the raw string into a *normalized* ingredient, the other into a *scientific name*.

## 7. Summary / findings worth investigating

| Dimension | scientific_name | odor | cuisine_normalize | ingredient_standardize |
|---|---|---|---|---|
| type | list | list | object | list |
| rows (per replica) | 4,280 | 3,792 | 116 | 17,017 |
| `status_code`=200 rows | all | all | all | all |
| `parsed = false` (≈) | 212 | 4 | 1 | 9 |
| input provenance | recipe ingredients (~74%) | nutrient × reaction | recipe cuisines (100%) | recipe ingredients (99.9%) |
| rows with non-empty output | ~2,444 | ~2,815 | 115 | ~16,639 |
| avg output length | 1.34 | 2.12 | — | 1.08 |
| distinct output values | ~2,346 | 50 terms (incl. ~23 off-vocab) | 207 countries / 32 sub-regions | ~3,586 |
| all-3 replicas identical | ~67% | ~36% | ~86% | ~83% |

**Findings worth investigating**

1. **`status_code` ≠ quality.** Every row returned HTTP `200`, yet a small but real share of answers fail strict JSON parsing (`parsed = false`) because the model *appends stray bytes* (a trailing quote + newline on `crab`), *returns two candidate answers* (`[]\n["fatty","nutty"]` on some `odor` rows), *truncates* an unclosed string, or emits an over-long list (cuisine `World`).
2. **Odour vocabulary drift** — the model was told to use a **27-term** lexicon but produced **50** terms (~23 off-vocabulary). Decide whether to keep/map them before feeding the odour column to a strict downstream model.
3. **Cuisine countries are not ISO-normalized** — only 166 of 207 country strings match `silver.countries`; colloquial spellings ("United States", "Ivory Coast", "Russia") would need a mapping pass.
4. **`odor` is the least reproducible task** (~36% identical across 3 replicas). If deterministic odour labels are needed, consider majority-voting across the three replicas.
5. **Nearly 60% of scientific-name inputs have no taxon** (empty list) — expected for processed / non-biological foods, but worth confirming none are false negatives (e.g. *spelt* correctly resolves, *fettuccine* does not).

## Cleanup — close the connection

In [17]:
# Close the connection to flush the WAL file (ICOCO.duckdb.wal)
%sql --close duckdb:////workspace/ICOCO.duckdb